# Compare performances of TTM on Buildings 900K

Chronos already tested, comparison

Simply modify the files_test paths and run all, to get results on buildings 900k desired test set.

In [1]:
import tempfile

import pandas as pd
from transformers import Trainer, TrainingArguments, set_seed
from transformers.integrations import INTEGRATION_TO_CALLBACK

from tsfm_public import TimeSeriesPreprocessor, get_datasets
from tsfm_public.toolkit.get_model import get_model

import datasets
import numpy as np

from gluonts.model.forecast import SampleForecast
from gluonts.model.evaluation import evaluate_forecasts
from gluonts.ev.metrics import MASE, MeanWeightedSumQuantileLoss
from gluonts.dataset.split import split

/home/joffreyma/miniforge3/envs/chronos_env/lib/python3.11/site-packages/gluonts/json.py:102: UserWarning: Using `json`-module for json-handling. Consider installing one of `orjson`, `ujson` to speed up serialization and deserialization.
  warnings.warn(


In [2]:
import warnings


# Suppress all warnings
warnings.filterwarnings("ignore")

In [3]:
TTM_MODEL_PATH = "ibm-granite/granite-timeseries-ttm-r2"
# Set seed for reproducibility
SEED = 41
set_seed(SEED)

# Results dir
OUT_DIR = "ttm_finetuned_models/"
dataset_name = "buildings_900K"

context_length=512 
forecast_length=64
batch_size=64

pd_batch_size = 1000

# Dataset
timestamp_column = "timestamp"
id_columns = ["item_id"]  # mention the ids that uniquely identify a time-series.

target_columns = ["target"]
split_config = {
    "train": [
        0,
        4000,
    ],
    "valid": [
        4000,
        8761-forecast_length,
    ],
    "test": [
        8761-forecast_length, # the context length is included by the splitter
        8761,
    ],
}
# Understanding the split config -- slides

column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": id_columns,
    "target_columns": target_columns,
    "control_columns": [],
}


In [4]:
files_test = ["../../../chronos-forecasting/data/buildings_900K_chronos_split_ttm/in_domain/data-00000-of-00001.arrow"]
ds_test = datasets.load_dataset(
    "arrow", data_files={'train': files_test}, split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
features = ds_test.features
features

{'item_id': Value(dtype='string', id=None),
 'start': Value(dtype='timestamp[s]', id=None),
 'freq': Value(dtype='string', id=None),
 'target': Sequence(feature=Value(dtype='float32', id=None), length=-1, id=None),
 'building_type': Value(dtype='string', id=None),
 'weather_year': Value(dtype='string', id=None),
 'broader_region': Value(dtype='string', id=None),
 'puma': Value(dtype='string', id=None),
 'building_id': Value(dtype='string', id=None),
 'group': Value(dtype='string', id=None),
 'timestamp': Sequence(feature=Value(dtype='timestamp[us]', id=None), length=-1, id=None)}

In [6]:
df_test = ds_test.to_pandas(batch_size=pd_batch_size, batched=True)
df_test

<generator object Dataset.to_pandas.<locals>.<genexpr> at 0x77183c547cd0>

In [7]:
forecasts = []
gluonds_test = []

In [8]:
tsp = TimeSeriesPreprocessor(
    **column_specifiers,
    context_length=context_length,
    prediction_length=forecast_length,
    scaling=True,
    encode_categorical=False,
    scaler_type="standard",
)

# Load model
zeroshot_model = get_model(
    TTM_MODEL_PATH,
    context_length=context_length,
    prediction_length=forecast_length,
    freq_prefix_tuning=False,
    freq=None,
    prefer_l1_loss=False,
    prefer_longer_context=True,
)

temp_dir = tempfile.mkdtemp()
# zeroshot_trainer
zeroshot_trainer = Trainer(
    model=zeroshot_model,
    args=TrainingArguments(
        output_dir=temp_dir,
        per_device_eval_batch_size=batch_size,
        seed=SEED,
        report_to="none",
    ),
)

for sub_df_test in df_test:
    sub_df_test_exploded = sub_df_test.explode(["target", "timestamp"], ignore_index=True)
    big_stride = len(sub_df_test_exploded) # Big stride to get only one window, it's the first but it's ok since the test is cut on the last segment of the time series anyway

    dset_train, dset_valid, dset_test = get_datasets(
        tsp, sub_df_test_exploded, split_config, use_frequency_token=zeroshot_model.config.resolution_prefix_tuning, stride=big_stride,
    )

    del sub_df_test_exploded

    # get predictions

    predictions_dict = zeroshot_trainer.predict(dset_test)

    predictions_np = predictions_dict.predictions[0]

    print(predictions_np.shape)

    # evaluate = zero-shot performance

    last_predictions = predictions_dict.predictions[0].squeeze(2)
    last_predictions = np.expand_dims(last_predictions, axis=1)

    start_date = dset_test.datasets[0][0]['timestamp']
    start_date = start_date.to_period(freq='H') + 1

    forecasts += [SampleForecast(samples=last_prediction, start_date=start_date) for last_prediction in last_predictions]
    
    gluonds_test += [{'start': pd.Period(ds_windows[0]['timestamp'], 'h') - 512 + 1, 'target': np.array(ds_windows.X.to_numpy().flatten(), dtype=np.float32)} for ds_windows in dset_test.datasets]



INFO:p-1306816:t-130953405572224:get_model.py:get_model:Loading model from: ibm-granite/granite-timeseries-ttm-r2
INFO:p-1306816:t-130953405572224:get_model.py:get_model:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = main.
INFO:p-1306816:t-130953405572224:get_model.py:get_model:[TTM] context_length = 512, prediction_length = 96


(1000, 64, 1)


(1000, 64, 1)


(1000, 64, 1)


(1000, 64, 1)


...

(1000, 64, 1)


(1000, 64, 1)


(1000, 64, 1)


(1000, 64, 1)


(1000, 64, 1)


In [9]:
# Split dataset for evaluation
_, test_template = split(gluonds_test, offset=-forecast_length)
test_data = test_template.generate_instances(forecast_length, windows=1)

metrics = (
    evaluate_forecasts(
        forecasts,
        test_data=test_data,
        metrics=[
            MASE(),
            MeanWeightedSumQuantileLoss(np.arange(0.1, 1.0, 0.1)),
        ],
        batch_size=batch_size,
    )
    .reset_index(drop=True)
    .to_dict(orient="records")
)
metrics

59998it [00:20, 2916.63it/s]


[{'MASE[0.5]': 1.1416847519337554,
  'mean_weighted_sum_quantile_loss': 0.5829667338751285}]

# Results

zero-shot on buildings_900K domain-shift
[{'MASE[0.5]': 1.144990696020555,
  'mean_weighted_sum_quantile_loss': 0.5852008666260413}]

zero-shot on buildings_900K in-domain
[{'MASE[0.5]': 1.1416847519337554,
  'mean_weighted_sum_quantile_loss': 0.5829667338751285}]